# All-method embedding distillation

Notebook này chạy tuần tự cả 8 method trong repo với **một cặp teacher → student
chọn ở cell 1** (`PAIR`). Ba cặp được cấu hình sẵn:

| `PAIR` | Teacher | Student | Câu hỏi |
| --- | --- | --- | --- |
| `qwen3_0.6b_to_minilm_h384` | `Qwen/Qwen3-Embedding-0.6B` (1024-d, last-token) | `MiniLMv2-L6-H384` (384-d, 6 layer) | student nhỏ, ít flow step |
| `bge_m3_to_minilm_h768` | `BAAI/bge-m3` (XLM-R, 1024-d, CLS) | `MiniLMv2-L6-H768` (768-d, 6 layer) | khác teacher family |
| `qwen3_4b_to_bert_base` | `Qwen/Qwen3-Embedding-4B` (2560-d, last-token) | `bert-base-uncased` (768-d, 12 layer) | capacity gap lớn |

- **Methods:** SimCSE-only, TALAS, GeoODE-KD, RKD, CDM, DSKD, EMO, Stella

Mỗi cặp mang theo pooling của teacher (`--teacher_pooling`: Qwen3 đọc token cuối,
BGE-M3 đọc CLS) và marker sub-word của tokenizer teacher (`--teacher_special_token`:
`Ġ` cho byte-level BPE của Qwen3, `▁` cho SentencePiece của BGE-M3) — cả hai được
truyền cho mọi method nên không phải chỉnh tay. Cache teacher được đặt tên theo cặp
và có metadata (teacher, pooling, corpus) nên chạy nhầm cache của teacher khác sẽ
bị từ chối ngay lúc khởi động.

**SimCSE-only** và **RKD** là hai mốc để đọc các method còn lại. SimCSE-only là control không distill: cùng student, cùng corpus, cùng lịch train và cùng pooling, chỉ bỏ phần teacher và giữ lại đúng InfoNCE mà các objective kia vốn đã chứa — nên phần điểm một method vượt lên trên dòng này chính là thứ tín hiệu teacher mang lại. **RKD** (Park et al., 2019) là mốc quan hệ: teacher chỉ giám sát khoảng cách và góc giữa các mẫu ở layer cuối, không nói gì về đường đi qua từng layer.

Mỗi method chạy trong một Python process riêng, có checkpoint, metrics và log riêng. Notebook dùng batch size thận trọng cho các method phải giữ teacher trên GPU; có thể tăng ở cell cấu hình nếu GPU còn nhiều VRAM. Nên dùng GPU hỗ trợ BF16; cặp Qwen3-4B cần ít nhất 24 GiB VRAM, hai cặp còn lại nhẹ hơn nhiều. Hai GPU sẽ được codebase tự động chia teacher/student.

TALAS, GeoODE-KD và RKD còn ghi thêm `depth_metrics.jsonl` (profile theo từng layer), đo bằng cùng một probe không tham số nên so sánh trực tiếp được. Cell 8 dựng các figure per-depth và figure so sánh giữa các method từ file này. SimCSE-only không có teacher embedding để đối chiếu nên không có profile này.


In [ ]:
# 1. Cấu hình thí nghiệm. Chỉnh các giá trị trong cell này trước khi chạy.
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"

# Ba cặp teacher -> student của bài. Mỗi cặp mang theo hai thứ phụ thuộc vào
# *family* của teacher, để không method nào phải chỉnh tay:
#   teacher_pooling:       cách đọc sentence vector của teacher. Qwen3-Embedding là
#                          decoder nên đọc token cuối; BGE-M3 là encoder XLM-R, đọc CLS.
#   teacher_special_token: marker sub-word mà CDM strip trước khi so token string
#                          ("Ġ" byte-level BPE của Qwen3, "▁" SentencePiece của BGE-M3).
#                          EMO đọc giá trị này như BOS token của teacher; BGE-M3 có
#                          "<s>" thật, Qwen3 không có BOS nên EMO giữ default của nó.
PAIRS = {
    "qwen3_0.6b_to_minilm_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "jim12345/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 12,
        "note": "student nhỏ (384-d, 6 layer): GeoODE chỉ có 6 flow step",
    },
    "bge_m3_to_minilm_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Large",
        "teacher_pooling": "cls",
        "teacher_special_token": "▁",
        "emo_teacher_special_token": "<s>",
        "min_vram_gib": 12,
        "note": "khác teacher family (XLM-R encoder, CLS pooling, SentencePiece)",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 24,
        "note": "capacity gap lớn (2560-d teacher 4B params -> 768-d student)",
    },
}
PAIR = "qwen3_4b_to_bert_base"  # <- chọn 1 trong 3 key của PAIRS

PAIR_CONFIG = PAIRS[PAIR]
TEACHER_MODEL = PAIR_CONFIG["teacher"]
STUDENT_MODEL = PAIR_CONFIG["student"]
TEACHER_POOLING = PAIR_CONFIG["teacher_pooling"]
TEACHER_SPECIAL_TOKEN = PAIR_CONFIG["teacher_special_token"]
EMO_TEACHER_SPECIAL_TOKEN = PAIR_CONFIG["emo_teacher_special_token"]
TRAIN_DATA_REL = Path("data/train_set/merged_3_data_5k_each.csv")
MAX_LENGTH = 256
EPOCHS = 5
NUM_WORKERS = 2
CUDA_VISIBLE_DEVICES = "0,1"
STOP_ON_ERROR = True
# Một công tắc cho mọi bước đánh giá và hiệu chỉnh.
#   False: eval từng epoch trên validation, ngưỡng pair sweep trên validation,
#          điểm test cuối vẫn là held-out.
#   True:  eval từng epoch trên test và sweep ngưỡng ngay trên test, bỏ hẳn
#          validation. Bám sát mục tiêu và nhanh hơn, nhưng không còn con số nào
#          là held-out — mọi lựa chọn epoch/ngưỡng đều đã nhìn thấy test.
EVAL_ON_TEST_EACH_EPOCH = True
PAIR_THRESHOLD_SOURCE = "test" if EVAL_ON_TEST_EACH_EPOCH else "validation"
# Chu kỳ eval từng epoch: N = eval sau mỗi N epoch, 0 = tắt hẳn (chỉ eval test cuối).
EVAL_EVERY = 0
SAVE_TO_GOOGLE_DRIVE = True

# Preset ưu tiên chạy được trên GPU 24-40 GiB với cặp nặng nhất (Qwen3-4B); hai cặp
# còn lại nhẹ hơn nhiều nên có thể tăng batch của cdm/dskd/emo/stella. TALAS/GeoODE/
# RKD cache teacher rồi giải phóng teacher nên không phụ thuộc kích thước teacher.
# Thứ tự dict cũng là thứ tự chạy: simcse chạy đầu vì nó không tải teacher và không
# cần cache, nên nếu data/eval có vấn đề thì hỏng ngay trong vài phút thay vì sau
# khi đã cache xong teacher; talas/geoode/rkd đứng liền nhau để dùng chung một
# lần chạy teacher. Batch size của rkd và simcse là số negative/quan hệ trong
# batch, không chỉ là nút chỉnh bộ nhớ: cả hai đo mọi thứ theo cặp trong batch.
METHOD_SETTINGS = {
    "simcse": {"batch_size": 32,  "learning_rate": 2e-5},
    "talas":  {"batch_size": 32, "learning_rate": 2e-5},
    "geoode": {"batch_size": 32, "learning_rate": 2e-5},
    "rkd":    {"batch_size": 32,  "learning_rate": 2e-5},
    "cdm":    {"batch_size": 32,  "learning_rate": 2e-5},
    "dskd":   {"batch_size": 32,  "learning_rate": 2e-5},
    "emo":    {"batch_size": 32,  "learning_rate": 1e-5},
    "stella": {"batch_size": 32,  "learning_rate": 5e-5},
}

RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{PAIR}_all_methods_{RUN_STAMP}"
print(f"Pair: {PAIR} — {PAIR_CONFIG['note']}")
print(f"  teacher: {TEACHER_MODEL} (pooling={TEACHER_POOLING}, marker={TEACHER_SPECIAL_TOKEN!r})")
print(f"  student: {STUDENT_MODEL}")
print(f"Run name: {RUN_NAME}")
print(f"Methods: {', '.join(METHOD_SETTINGS)}")


In [ ]:
# 2. Dùng repo hiện tại nếu notebook nằm trong repo; nếu không thì clone từ GitHub.
import subprocess
import sys

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file():
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if PROJECT_DIR.exists():
        assert (PROJECT_DIR / "main.py").is_file(), (
            f"Thư mục đã tồn tại nhưng không phải repo hợp lệ: {PROJECT_DIR}"
        )
        print(f"Reuse existing clone: {PROJECT_DIR}")
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

assert (PROJECT_DIR / "requirements.txt").is_file()
print(f"Project directory: {PROJECT_DIR}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies installed.")


In [ ]:
# 3. Chọn nơi lưu output và kiểm tra GPU/data trước khi tải model lớn.
import os

import torch

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Không tìm thấy training data: {TRAIN_DATA}"
for split in ("train_set", "val_set", "test_set"):
    split_dir = PROJECT_DIR / "data" / split
    assert split_dir.is_dir() and any(split_dir.glob("*.csv")), (
        f"Thiếu dữ liệu evaluation: {split_dir}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(f"{TEACHER_MODEL} cần GPU; hãy bật GPU runtime rồi chạy lại.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "Config của repo tải teacher bằng BF16 nhưng GPU hiện tại không hỗ trợ BF16. "
        "Hãy chọn A100, L4 hoặc GPU Ampere/newer."
    )

RUN_ROOT.mkdir(parents=True, exist_ok=False)
print(f"PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}")
print(f"Visible GPUs before subprocess filtering: {torch.cuda.device_count()}")
largest_gib = 0.0
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    largest_gib = max(largest_gib, props.total_memory / 2**30)
    print(f"  cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
if largest_gib < PAIR_CONFIG["min_vram_gib"]:
    print(
        f"[WARN] Cặp {PAIR} được ước lượng cần >= {PAIR_CONFIG['min_vram_gib']} GiB "
        f"trên một GPU cho các method online; GPU lớn nhất hiện có {largest_gib:.1f} GiB."
    )
print(f"Training data: {TRAIN_DATA}")
print(f"Output root: {RUN_ROOT}")


In [ ]:
# 4. Tạo và hiển thị command của từng method để kiểm tra trước khi train.
import shlex


def build_command(method, settings):
    method_dir = RUN_ROOT / method
    command = [
        sys.executable,
        str(PROJECT_DIR / "main.py"),
        "--method", method,
        "--train_data", str(TRAIN_DATA),
        "--student_model", STUDENT_MODEL,
        # TEACHER_MODEL cũng được truyền cho simcse: method này không tải teacher,
        # nhưng config ghi lại nó để run tự nói rõ mình là control của so sánh nào.
        "--teacher_model", TEACHER_MODEL,
        "--batch_size", str(settings["batch_size"]),
        "--epochs", str(EPOCHS),
        "--save_every", str(EPOCHS),
        "--lr", str(settings["learning_rate"]),
        "--max_length", str(MAX_LENGTH),
        "--save_dir", str(method_dir),
        "--num_workers", str(NUM_WORKERS),
        "--pair_threshold_source", PAIR_THRESHOLD_SOURCE,
        "--eval_every", str(EVAL_EVERY),
        "--no_wandb",
    ]
    if EVAL_ON_TEST_EACH_EPOCH:
        command.append("--evaluate_test_each_epoch")
    if method != "simcse":
        # Không có teacher thì không có pooling nào để chọn. Pooling đi theo family
        # của teacher (cell 1) và được mọi method áp dụng như nhau: talas/geoode/rkd
        # lúc cache, cdm/dskd/emo/stella ở từng step.
        command.extend(["--teacher_pooling", TEACHER_POOLING])
    if method == "cdm":
        # Marker sub-word của tokenizer teacher, để DTW so token string đúng.
        command.extend(["--teacher_special_token", TEACHER_SPECIAL_TOKEN])
    if method == "emo" and EMO_TEACHER_SPECIAL_TOKEN is not None:
        # EMO đọc flag này như BOS token của teacher (chỉ có nghĩa với encoder teacher).
        command.extend(["--teacher_special_token", EMO_TEACHER_SPECIAL_TOKEN])
    if method in ("talas", "geoode", "rkd"):
        # All three cache the same teacher pooling over the same corpus, so they
        # share one cache file instead of running the teacher three times. The file
        # is named after the pair and carries metadata, so a cache built for another
        # teacher is refused at startup instead of silently trained against.
        command.extend(["--cache_path", str(RUN_ROOT / "cache" / f"{PAIR}_teacher_train.pt")])
    return command


COMMANDS = {
    method: build_command(method, settings)
    for method, settings in METHOD_SETTINGS.items()
}
for method, command in COMMANDS.items():
    print(f"[{method.upper()}] {shlex.join(command)}\n")
print("Lưu ý: Stella dùng lịch mặc định 2 epoch stage 1 + 3 epoch stage 2.")
print("Lưu ý: simcse không tải teacher — nó là control không distill của bảng kết quả.")


In [ ]:
# 5. Chạy tuần tự tất cả method và tee stdout/stderr vào train.log.
import json
import time

# Notebook chỉ hiện dòng progress mới nhất, tối đa mỗi N giây, và cắt bớt độ dài.
# Log đầy đủ vẫn được ghi vào <method>_train.log.
PROGRESS_EVERY_SEC = 30
PROGRESS_MAX_CHARS = 160


def has_final_test(metrics_path):
    """True only for the end-of-run record.

    With EVAL_ON_TEST_EACH_EPOCH and EVAL_EVERY > 0 every epoch also writes a "test"
    payload, so the end-of-run record is identified by what it lacks: it carries no
    "train" block.
    """
    if not metrics_path.is_file():
        return False
    with metrics_path.open(encoding="utf-8") as handle:
        return any(
            json.loads(line).get("test") and not json.loads(line).get("train")
            for line in handle
            if line.strip()
        )


env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
env["TOKENIZERS_PARALLELISM"] = "false"
env["WANDB_MODE"] = "disabled"
# tqdm ghi mỗi ~0.1s một dòng "\r..." vào pipe; notebook in hết sẽ rất lag.
# Giảm tần suất tqdm từ phía subprocess, và bên dưới chỉ in progress mới nhất.
env["TQDM_MININTERVAL"] = str(PROGRESS_EVERY_SEC)
run_status = []


def stream_output(stream, log_handle):
    """Tee subprocess output: full log ra file, notebook chỉ hiện progress mới nhất.

    tqdm tách các lần cập nhật bằng "\r" thay vì "\n", nên đọc theo từng đoạn
    và tự tách theo cả hai ký tự. Dòng progress (chứa "%|") được ghi đè tại
    chỗ và chỉ in tối đa mỗi PROGRESS_EVERY_SEC giây; dòng thường in ngay.
    """
    buffer = ""
    last_progress = 0.0
    progress_shown = False
    while True:
        chunk = stream.read(4096)
        if not chunk:
            break
        buffer += chunk
        parts = buffer.replace("\r\n", "\n").replace("\r", "\n").split("\n")
        buffer = parts.pop()
        for line in parts:
            log_handle.write(line + "\n")
            if "%|" in line:
                now = time.perf_counter()
                if now - last_progress >= PROGRESS_EVERY_SEC:
                    print("\r" + line[:PROGRESS_MAX_CHARS].ljust(PROGRESS_MAX_CHARS), end="", flush=True)
                    last_progress = now
                    progress_shown = True
            elif line.strip():
                if progress_shown:
                    print()
                    progress_shown = False
                print(line)
        log_handle.flush()
    if buffer:
        log_handle.write(buffer + "\n")
        if "%|" not in buffer:
            print(buffer)
    if progress_shown:
        print()

for position, (method, command) in enumerate(COMMANDS.items(), start=1):
    method_dir = RUN_ROOT / method
    metrics_path = method_dir / "metrics.jsonl"
    log_path = RUN_ROOT / f"{method}_train.log"
    if has_final_test(metrics_path):
        print(f"[SKIP] {method.upper()} đã có final test: {metrics_path}")
        run_status.append({"method": method, "status": "skipped_complete", "seconds": 0.0})
        continue
    if metrics_path.exists():
        raise RuntimeError(
            f"{method} có run dở dang tại {method_dir}. "
            "Dùng RUN_NAME mới để tránh nối metrics vào run cũ."
        )

    print("\n" + "#" * 88)
    print(f"METHOD {position}/{len(COMMANDS)}: {method.upper()}")
    print(f"Log: {log_path}")
    print("#" * 88)
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        assert process.stdout is not None
        stream_output(process.stdout, log_handle)
        return_code = process.wait()
    elapsed = time.perf_counter() - started
    status = "complete" if return_code == 0 and has_final_test(metrics_path) else "failed"
    run_status.append({"method": method, "status": status, "seconds": elapsed})
    print(f"[{status.upper()}] {method} in {elapsed / 60:.1f} minutes")
    if status == "failed" and STOP_ON_ERROR:
        raise RuntimeError(f"Method {method} failed; xem log tại {log_path}")

print("\nRun status:")
for item in run_status:
    print(f"  {item['method']:8s} {item['status']:18s} {item['seconds'] / 60:8.1f} min")


In [ ]:
# 6. Tổng hợp validation theo epoch, final test và chi tiết từng benchmark.
import pandas as pd
from IPython.display import display


def benchmark_name(path, split):
    name = Path(path).stem
    suffix = f"_{split}"
    return name[:-len(suffix)] if name.endswith(suffix) else name


def detailed_rows(method, split, epoch, payload):
    rows = []
    primary_metrics = {
        "classification": "f1",
        "pair": "average_precision",
        "sts": "spearman",
    }
    for family, metric_name in primary_metrics.items():
        for path, raw_values in payload.get(family, {}).items():
            values = {"spearman": raw_values} if family == "sts" else dict(raw_values)
            rows.append({
                "method": method,
                "split": split,
                "epoch": epoch,
                "family": family,
                "benchmark": benchmark_name(path, split),
                "primary_metric": metric_name,
                "primary_score": float(values[metric_name]),
                **{key: float(value) for key, value in values.items() if isinstance(value, (int, float))},
            })
    return rows


epoch_summary_rows = []
final_summary_rows = []
detail_rows = []
for method in METHOD_SETTINGS:
    metrics_path = RUN_ROOT / method / "metrics.jsonl"
    if not metrics_path.is_file():
        print(f"[WARN] Missing metrics for {method}: {metrics_path}")
        continue
    with metrics_path.open(encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    for record in records:
        train = record.get("train")
        split = "validation" if record.get("validation") else "test"
        payload = record.get(split)
        if not payload:
            continue
        summary = payload["summary"]
        if train is not None:
            # Per-epoch record: on the split this run was configured to evaluate.
            epoch = int(train.get("epoch", 0))
            epoch_summary_rows.append({
                "method": method,
                "split": split,
                "stage": record.get("stage"),
                "epoch": epoch,
                "train_loss": train.get("loss"),
                **{key: float(summary[key]) for key in ("avg_iod", "avg_ood", "avg_all")},
            })
            detail_rows.extend(detailed_rows(method, split, epoch, payload))
        else:
            # End-of-run record: no "train" block.
            final_summary_rows.append({
                "method": method,
                **{key: float(summary[key]) for key in ("avg_iod", "avg_ood", "avg_all")},
            })
            detail_rows.extend(detailed_rows(method, "test", None, payload))

eval_by_epoch = pd.DataFrame(epoch_summary_rows)
final_test_results = pd.DataFrame(final_summary_rows)
benchmark_details = pd.DataFrame(detail_rows)
assert not final_test_results.empty, "Chưa có final test result nào để tổng hợp."
eval_by_epoch.to_csv(RUN_ROOT / "eval_by_epoch.csv", index=False)
final_test_results.to_csv(RUN_ROOT / "final_test_results.csv", index=False)
benchmark_details.to_csv(RUN_ROOT / "benchmark_details.csv", index=False)
pd.DataFrame(run_status).to_csv(RUN_ROOT / "run_status.csv", index=False)

epoch_split = eval_by_epoch["split"].iloc[0].upper() if not eval_by_epoch.empty else "-"
print(f"{epoch_split} BY EPOCH")
display(eval_by_epoch.style.format(precision=4))
print("FINAL TEST COMPARISON")
display(final_test_results.sort_values("avg_all", ascending=False).style.format(precision=4))
print(f"Saved summary files to: {RUN_ROOT}")


In [ ]:
# 7. Vẽ comparison plots và lưu PNG cùng kết quả.
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
if not eval_by_epoch.empty:
    split_label = eval_by_epoch["split"].iloc[0]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for method, frame in eval_by_epoch.groupby("method", sort=False):
        frame = frame.sort_values("epoch")
        ax.plot(frame["epoch"], frame["avg_all"], marker="o", label=method.upper())
    ax.set(title=f"{split_label.capitalize()} average across all benchmarks",
           xlabel="epoch", ylabel="avg_all")
    ax.legend(ncol=3, frameon=False)
    fig.tight_layout()
    fig.savefig(RUN_ROOT / "eval_avg_all.png", dpi=180, bbox_inches="tight")
    plt.show()

ordered = final_test_results.sort_values("avg_all", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(ordered["method"].str.upper(), ordered["avg_all"], color="#2a78d6")
ax.bar_label(ax.containers[0], fmt="%.4f", padding=4)
ax.set(title="Final test comparison", xlabel="average score across all benchmarks", ylabel="")
ax.set_xlim(0, min(1.0, float(ordered["avg_all"].max()) * 1.15))
fig.tight_layout()
fig.savefig(RUN_ROOT / "final_test_comparison.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# 8. Phân tích per-depth cho các method có depth_metrics.jsonl (geoode, talas, rkd).
from IPython.display import Image

depth_runs = sorted(
    method_dir
    for method_dir in RUN_ROOT.iterdir()
    if method_dir.is_dir() and (method_dir / "depth_metrics.jsonl").is_file()
)
if not depth_runs:
    print("Chưa có depth_metrics.jsonl. Train lại với --depth_log_every > 0.")
else:
    print("Depth diagnostics:", ", ".join(run.name for run in depth_runs))
    analysis = subprocess.run(
        [
            sys.executable,
            str(PROJECT_DIR / "scripts" / "plot_depth_diagnostics.py"),
            *[str(run) for run in depth_runs],
            "--out",
            str(RUN_ROOT / "figures"),
        ],
        capture_output=True,
        text=True,
    )
    print(analysis.stdout)
    if analysis.returncode != 0:
        print(analysis.stderr)

    figures = sorted((RUN_ROOT / "figures").glob("*.png"))
    # So sánh giữa các method trước, rồi mới tới chi tiết từng method.
    for figure in sorted(figures, key=lambda path: (not path.name.startswith("comparison"), path.name)):
        print(figure.name)
        display(Image(filename=str(figure)))


In [ ]:
# 9. Tuỳ chọn: đóng gói toàn bộ run thành một file ZIP.
import shutil

archive_path = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
print(f"Created archive: {archive_path}")
if IN_COLAB and not SAVE_TO_GOOGLE_DRIVE:
    from google.colab import files

    files.download(archive_path)
